In [1]:
import numpy as np
from transformers import pipeline
print("Packaged imported successfully")

d:\dream_machine\AI-content-moderator\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Packaged imported successfully


In [35]:
# -------------------------------
# 1. Load Model (singleton)
# -------------------------------
class ModerationModel:
    def __init__(self):
        self.model = pipeline(
            "text-classification",
            model="unitary/toxic-bert"
        )

    def predict(self, text: str) -> dict:
        result = self.model(text)[0]
        label = result["label"].lower()
        score = result["score"]

        if label == "toxic":
            toxicity = score
        else:
            toxicity = 1 - score

        return {
            "toxicity": toxicity,
            "hate": 0.0,
            "sexual": 0.0,
            "self_harm": 0.0
        }

moderation_model = ModerationModel()



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7047.31it/s]
BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:

# -------------------------------
# 2. Decision Logic
# -------------------------------
THRESHOLDS = {
    "toxicity": 0.8,
    "hate": 0.7,
    "sexual": 0.75,
    "self_harm": 0.85
}


def decide_moderation(scores: dict):
    for label, threshold in THRESHOLDS.items():
        if scores.get(label, 0) > threshold:
            return {
                "status": "BLOCKED",
                "reason": label
            }

    if scores.get("toxicity", 0) > 0.5:
        return {
            "status": "FLAGGED",
            "reason": "toxicity"
        }

    return {
        "status": "APPROVED",
        "reason": None
    }



In [37]:

# -------------------------------
# 3. Moderation Service
# -------------------------------
class ModerationService:

    def moderate(self, text: str) -> dict:
        scores = moderation_model.predict(text)
        decision = decide_moderation(scores)

        return {
            "scores": scores,
            "status": decision["status"],
            "reason": decision["reason"]
        }


moderation_service = ModerationService()



In [46]:

# -------------------------------
# 4. Test It
# -------------------------------
texts = [
    "i will kill you ",
    "I hate you",
    "You are stupid",
    "Let's build something amazing",
    "I will kill you",
]

for text in texts:
    result = moderation_service.moderate(text)
    print(f"Text: {text}")
    print(f"Result: {result}")
    print("-" * 50)

Text: i will kill you 
Result: {'scores': {'toxicity': 0.9071627855300903, 'hate': 0.0, 'sexual': 0.0, 'self_harm': 0.0}, 'status': 'BLOCKED', 'reason': 'toxicity'}
--------------------------------------------------
Text: I hate you
Result: {'scores': {'toxicity': 0.9509986639022827, 'hate': 0.0, 'sexual': 0.0, 'self_harm': 0.0}, 'status': 'BLOCKED', 'reason': 'toxicity'}
--------------------------------------------------
Text: You are stupid
Result: {'scores': {'toxicity': 0.9847338795661926, 'hate': 0.0, 'sexual': 0.0, 'self_harm': 0.0}, 'status': 'BLOCKED', 'reason': 'toxicity'}
--------------------------------------------------
Text: Let's build something amazing
Result: {'scores': {'toxicity': 0.0007403437048196793, 'hate': 0.0, 'sexual': 0.0, 'self_harm': 0.0}, 'status': 'APPROVED', 'reason': None}
--------------------------------------------------
Text: I will kill you
Result: {'scores': {'toxicity': 0.9071627855300903, 'hate': 0.0, 'sexual': 0.0, 'self_harm': 0.0}, 'status': 'B

In [39]:
print(moderation_model.model("I will kill you"))

[{'label': 'toxic', 'score': 0.9071627855300903}]
